# Evaluation — Trained Modules vs. Baselines

Loads `data/baselines/results.json` (written by `scripts/evaluate_all.py`
and `scripts/run_baselines.py`) and visualizes how the trained REXA modules
compare against the classical Keyword-Overlap and TF-IDF-Cosine baselines
on the star-prediction task.

Run the following first if `results.json` doesn't exist yet:

```bash
cd ml
python scripts/prepare_data.py
python scripts/train_sentence_roles.py
python scripts/train_concept_coverage.py
python scripts/train_support_contradiction.py
python scripts/train_reasoning_depth.py
python scripts/train_star_prediction.py
python scripts/evaluate_all.py
python scripts/run_baselines.py
```

In [ ]:
import json
from pathlib import Path

import pandas as pd

RESULTS_PATH = Path("../../data/baselines/results.json")
results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))

print("Top-level keys:", list(results.keys()))
module_metrics = results.get("evaluate_all", {}).get("modules", {})
pd.DataFrame(
    {
        module: {k: v for k, v in metrics.items() if not isinstance(v, dict)}
        for module, metrics in module_metrics.items()
    }
).T

## Star prediction: trained model vs. baselines

In [ ]:
comparison = results.get("evaluate_all", {}).get("star_prediction_vs_baselines", {})
comparison_df = pd.DataFrame(comparison).T
comparison_df = comparison_df[
    [c for c in ["mae", "rmse", "spearman_rho", "exact_accuracy", "within_one_star_accuracy", "quadratic_weighted_kappa"] if c in comparison_df.columns]
]
comparison_df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
comparison_df["mae"].plot(kind="bar", ax=ax, color=["#4c72b0", "#dd8452", "#55a868"])
ax.set_ylabel("Mean Absolute Error (stars)")
ax.set_title("Star prediction MAE: trained model vs. baselines (lower is better)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Standalone baseline sweep

`results.json` also stores a separate `baselines_comparison` block written
directly by `scripts/run_baselines.py`, evaluated over the *full* sample
dataset (train+val+test combined) rather than just the test split — useful
as a sanity check that the baselines behave consistently at a larger
sample size.

In [ ]:
full_sweep = results.get("baselines_comparison", {}).get("baselines", [])
pd.DataFrame(full_sweep).set_index("name")[
    ["mae", "rmse", "spearman_rho", "exact_accuracy", "within_one_star_accuracy"]
]